# Chapter 3: Convolutional Networks

Paired notebook for Chapter 3 of *Inductive Biases in Neural Networks*.
Trains the Chapter 1 MLP baseline and a small CNN on the UCI 8x8 optdigits
dataset, runs the **permuted-pixel control** as a falsification test of the
CNN's locality prior, gradient-checks `Conv2D`, and saves the chapter figure.
Pure-Python `scratchnn` throughout (the model and the experiment).

**Seeds:** model/shuffle seed `0` for both models; the fixed pixel
permutation uses `random.Random(seed + 100)`. Same seeds and hyperparameters
(40 epochs, lr 0.1, batch 32) as `examples/digits_cnn.py`, so MLP and CNN
are compared on equal footing.

**Figure saved:** `../book/figures/ch03-permuted-pixel.pdf`

**Headline numbers (regenerated below):** MLP 2410 params at 96.1% test;
CNN 1490 params at 95.4%; the conv kernel is 40 params versus 9360 for an
equivalent `Linear(64, 144)` (234x). Under a fixed pixel permutation the MLP
drops 0.45pp and the CNN drops 1.50pp, so the CNN loses more than three times
as much. `gradient_check` on `Conv2D` is on the order of 1e-9.

## Setup

The library is pure Python (`math`, `random` only). The dataset loader and
metrics come from `examples/digits.py`, which reads the UCI optdigits CSV
splits (3823 train, 1797 test; integer pixel counts 0-16 normalized to
[0, 1]).

In [1]:
import os
import sys
import random

# Use the scratchnn examples for the dataset loader and accuracy metric.
EXAMPLES = "/home/spinoza/github/repos/scratchnn/examples"
sys.path.insert(0, EXAMPLES)

import scratchnn as nn
from digits import load_digits, TRAIN, TEST, accuracy

X_tr, Y_tr = load_digits(TRAIN)
X_te, Y_te = load_digits(TEST)
print(f"train: {len(X_tr)} samples, test: {len(X_te)} samples")
print(f"input dim: {len(X_tr[0])} pixels, classes: {len(set(Y_tr))}")

train: 3823 samples, test: 1797 samples
input dim: 64 pixels, classes: 10


## Models

The MLP baseline is Chapter 1's `64 -> 32 -> 10`. The CNN is one convolutional
layer of four 3x3 kernels over the 8x8 single-channel image, a ReLU, and a
linear head reading the flattened `4 x 6 x 6 = 144` feature maps. The network
emits logits; `SoftmaxCrossEntropy` is the output head.

In [2]:
def build_mlp(hidden, seed):
    random.seed(seed)
    return nn.Network([
        nn.Linear(64, hidden),
        nn.Tanh(),
        nn.Linear(hidden, 10),
    ], nn.SoftmaxCrossEntropy())


def build_cnn(seed):
    random.seed(seed)
    return nn.Network([
        nn.Conv2D(in_channels=1, out_channels=4, kernel_size=3, in_h=8, in_w=8),
        nn.ReLU(),
        nn.Linear(4 * 6 * 6, 10),
    ], nn.SoftmaxCrossEntropy())


def count_params(model):
    return sum(len(values) for values, _ in model.parameters())


mlp_params = count_params(build_mlp(32, 0))
cnn_params = count_params(build_cnn(0))
print(f"MLP params: {mlp_params}")
print(f"CNN params: {cnn_params}")

MLP params: 2410
CNN params: 1490


### The parameter count, isolated

The CNN is smaller than the MLP, and almost all of its spatial capacity sits
in the 40-parameter conv kernel. To price the locality-plus-sharing
commitment directly, compare the conv layer against the `Linear(64, 144)`
that would produce the same 144-dimensional output without any weight
sharing.

In [3]:
conv_only = count_params(nn.Network([nn.Conv2D(1, 4, 3, 8, 8)],
                                     nn.SoftmaxCrossEntropy()))
linear_equiv = sum(len(v) for v, _ in nn.Linear(64, 144).parameters())
print(f"Conv2D(1, 4, k=3):      {conv_only} params")
print(f"Linear(64, 144):        {linear_equiv} params")
print(f"reduction:              {linear_equiv / conv_only:.0f}x")
print(f"CNN vs MLP total:       {cnn_params} vs {mlp_params} "
      f"({100 * (1 - cnn_params / mlp_params):.0f}% smaller)")

Conv2D(1, 4, k=3):      40 params
Linear(64, 144):        9360 params
reduction:              234x
CNN vs MLP total:       1490 vs 2410 (38% smaller)


## Gradient check on `Conv2D`

The hand-derived `Conv2D.backward` is verified against a central-difference
numerical gradient, the same check used in Chapter 1. The worst relative
error across parameters should sit near machine precision, well under the
1e-4 tolerance.

In [4]:
def gradient_check(net, x, y, eps=1e-5):
    net.zero_grad()
    net.forward(x)
    net.backward(y)
    worst = 0.0
    for values, grads in net.parameters():
        for k in range(len(values)):
            original = values[k]
            values[k] = original + eps
            loss_plus = net.loss_value(x, y)
            values[k] = original - eps
            loss_minus = net.loss_value(x, y)
            values[k] = original
            numerical = (loss_plus - loss_minus) / (2.0 * eps)
            analytical = grads[k]
            denom = max(abs(numerical) + abs(analytical), 1e-12)
            worst = max(worst, abs(numerical - analytical) / denom)
    return worst


random.seed(7)
gc_net = nn.Network([
    nn.Conv2D(in_channels=1, out_channels=2, kernel_size=3, in_h=5, in_w=5),
    nn.ReLU(),
    nn.Linear(2 * 3 * 3, 3),
], nn.SoftmaxCrossEntropy())
gc_x = [random.uniform(-1, 1) for _ in range(25)]
gc_worst = gradient_check(gc_net, gc_x, 1)
print(f"Conv2D worst relative gradient error: {gc_worst:.2e}")

Conv2D worst relative gradient error: 7.89e-10


## Training: standard pixels

Same dataset and training loop for both models: 40 epochs, learning rate 0.1,
batch size 32, mini-batch SGD. A few minutes each in pure Python.

In [5]:
EPOCHS, LR, BATCH, SEED = 40, 0.1, 32, 0

mlp = build_mlp(32, SEED)
mlp.fit(X_tr, Y_tr, epochs=EPOCHS, lr=LR, batch_size=BATCH)
mlp_std = accuracy(mlp, X_te, Y_te)
print(f"MLP  test accuracy (standard): {mlp_std:.4f}")

cnn = build_cnn(SEED)
cnn.fit(X_tr, Y_tr, epochs=EPOCHS, lr=LR, batch_size=BATCH)
cnn_std = accuracy(cnn, X_te, Y_te)
print(f"CNN  test accuracy (standard): {cnn_std:.4f}")

MLP  test accuracy (standard): 0.9610


CNN  test accuracy (standard): 0.9544


## The permuted-pixel control

Apply one fixed random permutation to the 64 input pixels and retrain both
models from scratch on the permuted data. The permutation destroys spatial
adjacency: a 3x3 kernel now sees a meaningless window of unrelated pixels. The
MLP has no spatial prior to lose, so it should be nearly unchanged. If the CNN
is genuinely using locality, it should lose more. The gap (and its direction)
is the falsification signature.

In [6]:
def permute_pixels(X, perm):
    return [[x[i] for i in perm] for x in X]


rng = random.Random(SEED + 100)
perm = list(range(64))
rng.shuffle(perm)
Xp_tr = permute_pixels(X_tr, perm)
Xp_te = permute_pixels(X_te, perm)

mlp_p = build_mlp(32, SEED)
mlp_p.fit(Xp_tr, Y_tr, epochs=EPOCHS, lr=LR, batch_size=BATCH)
mlp_perm = accuracy(mlp_p, Xp_te, Y_te)
print(f"MLP  test accuracy (permuted): {mlp_perm:.4f}")

cnn_p = build_cnn(SEED)
cnn_p.fit(Xp_tr, Y_tr, epochs=EPOCHS, lr=LR, batch_size=BATCH)
cnn_perm = accuracy(cnn_p, Xp_te, Y_te)
print(f"CNN  test accuracy (permuted): {cnn_perm:.4f}")

MLP  test accuracy (permuted): 0.9566


CNN  test accuracy (permuted): 0.9393


## Results

In [7]:
mlp_drop = 100 * (mlp_std - mlp_perm)
cnn_drop = 100 * (cnn_std - cnn_perm)

print(f"{'model':<18}{'pixels':<12}{'params':>8}{'test acc':>11}")
print("-" * 49)
print(f"{'MLP (64-32-10)':<18}{'standard':<12}{mlp_params:>8}{mlp_std:>11.4f}")
print(f"{'MLP (64-32-10)':<18}{'permuted':<12}{mlp_params:>8}{mlp_perm:>11.4f}")
print(f"{'CNN (1,4,k=3+L)':<18}{'standard':<12}{cnn_params:>8}{cnn_std:>11.4f}")
print(f"{'CNN (1,4,k=3+L)':<18}{'permuted':<12}{cnn_params:>8}{cnn_perm:>11.4f}")
print()
print(f"MLP drop under permutation: {mlp_drop:.2f} pp")
print(f"CNN drop under permutation: {cnn_drop:.2f} pp")
print(f"ratio (CNN drop / MLP drop): {cnn_drop / mlp_drop:.2f}x")
print(f"gradient_check (Conv2D):     {gc_worst:.2e}")

model             pixels        params   test acc
-------------------------------------------------
MLP (64-32-10)    standard        2410     0.9610
MLP (64-32-10)    permuted        2410     0.9566
CNN (1,4,k=3+L)   standard        1490     0.9544
CNN (1,4,k=3+L)   permuted        1490     0.9393

MLP drop under permutation: 0.45 pp
CNN drop under permutation: 1.50 pp
ratio (CNN drop / MLP drop): 3.37x
gradient_check (Conv2D):     7.89e-10


## Figure: the permuted-pixel comparison

Grouped bars, standard versus permuted test accuracy for both models. The CNN
bar pair separates visibly more than the MLP pair: the visual statement of the
falsification result. Saved to `../book/figures/ch03-permuted-pixel.pdf` and
placed in the chapter at the permuted-pixel control section.

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_PDF = "../book/figures/ch03-permuted-pixel.pdf"
labels = ["MLP\n(64-32-10)", "CNN\n(Conv+Linear)"]
standard = [100 * mlp_std, 100 * cnn_std]
permuted = [100 * mlp_perm, 100 * cnn_perm]
x = range(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(6.0, 4.0))
b1 = ax.bar([i - w / 2 for i in x], standard, w, label="standard pixels",
            color="#4C72B0")
b2 = ax.bar([i + w / 2 for i in x], permuted, w, label="permuted pixels",
            color="#C44E52")
ax.set_ylabel("test accuracy (%)")
ax.set_title("Permuted-pixel control: CNN loses more than the MLP")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylim(90, 98)
ax.legend(frameon=False, loc="lower left")
for bars in (b1, b2):
    for rect in bars:
        ax.annotate(f"{rect.get_height():.1f}",
                    (rect.get_x() + rect.get_width() / 2, rect.get_height()),
                    ha="center", va="bottom", fontsize=9,
                    xytext=(0, 1), textcoords="offset points")
fig.tight_layout()
os.makedirs(os.path.dirname(FIG_PDF), exist_ok=True)
fig.savefig(FIG_PDF)
print(f"saved {FIG_PDF}")

saved ../book/figures/ch03-permuted-pixel.pdf
